# 如何判断基金的高点和低点，尽量避免深套？

- 使用 KDJ 指标判断市场情绪，设计基金回撤策略
- 技术指标到底能不能帮助我们改善买卖节奏？

我分享一个最常见指标：**KDJ**。

核心问题：
**只用 KDJ，能否找到相对更合理的基金买卖时机，并控制回撤？**

> 风险提示：以下内容仅用于学习与研究，不构成投资建议。


## 代码章节
1. 数据准备
2. KDJ原理与图形
3. 交易信号规则
4. 回测与指标对比
5. 收益曲线与总结


In [31]:
import os
import warnings
from typing import Dict, Optional, List

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

CACHE_DIR = './data_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

pd.set_option('display.max_columns', 50)


## 1) 数据准备模块：`load_market_data(fund_code)`
数据来源优先级：
1. 本地 `data_cache/{fund_code}.csv`
2. 在线源（akshare / yfinance）兜底

并完成：
- 字段标准化（日期、开高低收量）
- 缺失值处理
- 异常处理（报错信息可读）


In [32]:
def _parse_volume(v):
    """Parse volume values like 12.3B / 560M / 120K into numeric float."""
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        return float(v)
    s = str(v).strip().replace(',', '')
    try:
        if s.endswith('B'):
            return float(s[:-1]) * 1e9
        if s.endswith('M'):
            return float(s[:-1]) * 1e6
        if s.endswith('K'):
            return float(s[:-1]) * 1e3
        return float(s)
    except Exception:
        return np.nan


def _standardize_ohlcv(raw: pd.DataFrame) -> pd.DataFrame:
    """Standardize raw market data to Date index + OHLCV columns.

    Returns columns: Open, High, Low, Close, Volume.
    """
    if raw is None or raw.empty:
        raise ValueError('raw data is empty')

    df = raw.copy()
    rename_map = {
        '日期': 'Date', 'date': 'Date', 'Date': 'Date',
        '开盘': 'Open', 'open': 'Open', 'Open': 'Open',
        '高': 'High', 'high': 'High', 'High': 'High',
        '低': 'Low', 'low': 'Low', 'Low': 'Low',
        '收盘': 'Close', 'close': 'Close', 'Close': 'Close',
        '交易量': 'Volume', 'volume': 'Volume', 'Volume': 'Volume',
    }
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})

    # Normalize datetime index.
    if 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
        df = df.dropna(subset=['Date']).set_index('Date')
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')
        df = df[~df.index.isna()]

    # Ensure required price columns exist and are numeric.
    for col in ['Open', 'High', 'Low', 'Close']:
        if col not in df.columns:
            raise ValueError(f'missing required column: {col}')
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'Volume' in df.columns:
        df['Volume'] = df['Volume'].apply(_parse_volume)
    else:
        df['Volume'] = np.nan

    # Sort, deduplicate, fill basic missing values.
    df = df.sort_index()
    df = df[~df.index.duplicated(keep='last')]
    df[['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].ffill()
    df['Volume'] = df['Volume'].fillna(0.0)
    df = df.dropna(subset=['Close'])

    if df.empty:
        raise ValueError('data is empty after cleaning')

    return df


def load_market_data(fund_code: str, start_date: str = '2018-01-01', end_date: Optional[str] = None) -> pd.DataFrame:
    """Load fund market data by code.

    Priority:
    1) local CSV under data_cache
    2) akshare
    3) yfinance
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime('%Y-%m-%d')

    errors: List[str] = []

    # 1) Local cache first for reproducible video demo.
    local_path = os.path.join(CACHE_DIR, f'{fund_code}.csv')
    if os.path.exists(local_path):
        try:
            local_raw = pd.read_csv(local_path)
            df = _standardize_ohlcv(local_raw)
            return df.loc[(df.index >= start_date) & (df.index <= end_date)].copy()
        except Exception as e:
            errors.append(f'local_cache: {e}')

    # 2) akshare fallback.
    if fund_code.isdigit() and len(fund_code) == 6:
        try:
            import akshare as ak
            raw = ak.fund_etf_hist_em(symbol=fund_code, period='daily', adjust='qfq')
            df = _standardize_ohlcv(raw)
            df = df.loc[(df.index >= start_date) & (df.index <= end_date)].copy()
            if not df.empty:
                return df
        except Exception as e:
            errors.append(f'akshare: {e}')

    # 3) yfinance final fallback.
    try:
        import yfinance as yf
        candidates = [fund_code]
        if fund_code.isdigit() and len(fund_code) == 6:
            candidates += [f'{fund_code}.SS', f'{fund_code}.SZ']

        for symbol in candidates:
            yf_df = yf.download(symbol, start=start_date, end=end_date, progress=False, auto_adjust=False)
            if yf_df is not None and not yf_df.empty:
                df = _standardize_ohlcv(yf_df.reset_index())
                if not df.empty:
                    return df
    except Exception as e:
        errors.append(f'yfinance: {e}')

    raise RuntimeError(f'Failed to load {fund_code}. Details: {errors}')

In [33]:
FUND_CODE = '510300'  # 可切换为 '513130'
START_DATE = '2018-01-01'
END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')

data = load_market_data(FUND_CODE, START_DATE, END_DATE)
print(f'Loaded {FUND_CODE}: {len(data)} rows | {data.index.min().date()} -> {data.index.max().date()}')
display(data.tail(10))


Loaded 510300: 1981 rows | 2018-01-02 -> 2026-03-06


,Close,Open,High,Low,Volume,涨跌幅
Date,,,,,,
2026-02-13,4.671,4.714,4.714,4.667,1.370000e+09,-1.18%
2026-02-24,4.716,4.725,4.736,4.705,6.242500e+08,0.96%
2026-02-25,4.750,4.717,4.778,4.716,8.734000e+08,0.72%
2026-02-26,4.735,4.751,4.754,4.712,7.663700e+08,-0.32%
2026-02-27,4.725,4.720,4.732,4.700,6.062500e+08,-0.21%
2026-03-02,4.739,4.694,4.743,4.674,1.240000e+09,0.30%
2026-03-03,4.678,4.741,4.751,4.665,9.665600e+08,-1.29%
2026-03-04,4.610,4.640,4.650,4.587,8.292200e+08,-1.45%
2026-03-05,4.654,4.647,4.678,4.639,5.684200e+08,0.95%


## 2) KDJ 指标详解模块
### 核心定义
- KDJ 用来衡量价格在最近区间中的相对位置与动能变化。
- **K线**：快线，对价格变化敏感。
- **D线**：慢线，对K进行平滑。
- **J线**：`J = 3K - 2D`，波动更大，因此更容易出现超买超卖。

### 为什么 J 线更容易超买超卖？
因为 J 是对 K 与 D 差值的放大表达，当 K 偏离 D 时，J 会放大偏离幅度。


In [34]:
def calculate_kdj(data: pd.DataFrame, n: int = 9, k_smooth: int = 3, d_smooth: int = 3) -> pd.DataFrame:
    """Calculate KDJ indicator (K, D, J).

    Parameters
    - n: lookback window for RSV
    - k_smooth / d_smooth: EMA-like smoothing factors
    """
    required = ['High', 'Low', 'Close']
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(f'missing columns for KDJ: {missing}')

    df = data.copy()

    # RSV = price position within recent high-low range.
    ll = df['Low'].rolling(window=n, min_periods=n).min()
    hh = df['High'].rolling(window=n, min_periods=n).max()
    spread = (hh - ll).replace(0, np.nan)
    rsv = (df['Close'] - ll) / spread * 100
    rsv = rsv.clip(0, 100)

    # Smooth RSV into K and D, then derive J.
    df['K'] = rsv.ewm(alpha=1 / k_smooth, adjust=False).mean()
    df['D'] = df['K'].ewm(alpha=1 / d_smooth, adjust=False).mean()
    df['J'] = 3 * df['K'] - 2 * df['D']

    return df


def plot_kdj_lines(df: pd.DataFrame, title: str) -> go.Figure:
    """Plot close price and KDJ lines in two interactive subplots."""
    required = ['Close', 'K', 'D', 'J']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'missing columns for KDJ plot: {missing}')

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07, row_heights=[0.55, 0.45])
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close', line=dict(color='#1f77b4')), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['K'], mode='lines', name='K', line=dict(color='#2ca02c')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['D'], mode='lines', name='D', line=dict(color='#ff7f0e')), row=2, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['J'], mode='lines', name='J', line=dict(color='#d62728')), row=2, col=1)

    # Typical KDJ visual guide lines.
    fig.add_hline(y=20, line_dash='dash', line_color='gray', row=2, col=1)
    fig.add_hline(y=80, line_dash='dash', line_color='gray', row=2, col=1)

    fig.update_layout(template='plotly_white', height=700, title=title, hovermode='x unified')
    fig.update_yaxes(title='Price', row=1, col=1)
    fig.update_yaxes(title='KDJ', row=2, col=1)
    return fig

In [35]:
df_kdj = calculate_kdj(data)
display(df_kdj[['Close', 'K', 'D', 'J']].tail(12))
plot_kdj_lines(df_kdj.dropna(), title=f'{FUND_CODE} - KDJ').show()


,Close,K,D,J
Date,,,,
2026-02-11,4.723,66.572661,54.955833,89.806316
2026-02-12,4.727,75.117704,61.676457,102.000200
2026-02-13,4.671,68.693188,64.015367,78.048830
2026-02-24,4.716,73.185640,67.072125,85.412670
2026-02-25,4.750,76.568204,70.237485,89.229644
2026-02-26,4.735,75.847057,72.107342,83.326486
2026-02-27,4.725,67.982122,70.732269,62.481829
2026-03-02,4.739,66.943036,69.469191,61.890726
2026-03-03,4.678,48.463499,62.467294,20.455910


## 3) KDJ 交易信号模块
策略定义（仅 KDJ，共 3 套）：

1. **原策略**
   - 买入：K 上穿 D（金叉）且 K < 20
   - 卖出：K 下穿 D（死叉）且 K > 60

2. **策略 A**
   - 买入：K 上穿 D（金叉）且 K < 20
   - 卖出：K 下穿 D（死叉）且 K > 50

3. **策略 B**
   - 买入：K 上穿 D（金叉）且 K < 15
   - 卖出：K 下穿 D（死叉）且 K > 60

说明：
- 所有策略都使用同一套状态约束（空仓才买、持仓才卖）。
- 回测阶段统一使用同一交易佣金参数。

In [36]:
def generate_trade_signals(df: pd.DataFrame, buy_th: float = 20, sell_th: float = 60) -> pd.DataFrame:
    """Generate KDJ trade signals from configurable thresholds.

    Rules
    - Buy signal (1): K golden-crosses D and K < buy_th
    - Sell signal (-1): K death-crosses D and K > sell_th
    """
    required = ['K', 'D']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'missing columns for signal generation: {missing}')

    out = df.copy()
    prev_k = out['K'].shift(1)
    prev_d = out['D'].shift(1)

    # Cross detections based on yesterday vs today.
    golden = (prev_k <= prev_d) & (out['K'] > out['D'])
    death = (prev_k >= prev_d) & (out['K'] < out['D'])

    buy = golden & (out['K'] < buy_th)
    sell = death & (out['K'] > sell_th)

    out['signal'] = 0
    out.loc[buy, 'signal'] = 1
    out.loc[sell, 'signal'] = -1

    return out


def build_strategy_signals(df_kdj: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """Build all strategy signal DataFrames for comparison.

    Included KDJ strategies (same commission rules in backtest):
    1) Original: K<20 / K>60
    2) Strategy A: K<20 / K>50
    3) Strategy B: K<15 / K>60
    """
    return {
        'Original (K<20 / K>60)': generate_trade_signals(df_kdj, buy_th=20, sell_th=60),
        'Strategy A (K<20 / K>50)': generate_trade_signals(df_kdj, buy_th=20, sell_th=50),
        'Strategy B (K<15 / K>60)': generate_trade_signals(df_kdj, buy_th=15, sell_th=60),
    }


def plot_signals(df: pd.DataFrame, title: str) -> go.Figure:
    """Plot candlestick + close line + buy/sell markers."""
    required = ['Open', 'High', 'Low', 'Close', 'signal']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'missing columns for signal plot: {missing}')

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df.index, open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
        name='Candles', increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))
    fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Close', line=dict(color='#1f77b4')))

    buy_df = df[df['signal'] == 1]
    sell_df = df[df['signal'] == -1]

    fig.add_trace(go.Scatter(x=buy_df.index, y=buy_df['Low'], mode='markers', name='Buy', marker=dict(symbol='triangle-up', size=11, color='#2e7d32')))
    fig.add_trace(go.Scatter(x=sell_df.index, y=sell_df['High'], mode='markers', name='Sell', marker=dict(symbol='triangle-down', size=11, color='#c62828')))

    fig.update_layout(template='plotly_white', height=620, title=title, hovermode='x unified')
    return fig

In [37]:
strategy_signal_map = build_strategy_signals(df_kdj)

for name, sig_df in strategy_signal_map.items():
    tmp = sig_df.dropna().copy()
    print(f'\n{name}')
    print(tmp['signal'].value_counts().sort_index())
    display(tmp[['Close', 'K', 'D', 'J', 'signal']].tail(12))
    plot_signals(tmp, title=f'{FUND_CODE} - {name} Signals').show()


Original (K<20 / K>60)
signal
-1      92
 0    1871
 1      10
Name: count, dtype: int64


,Close,K,D,J,signal
Date,,,,,
2026-02-11,4.723,66.572661,54.955833,89.806316,0
2026-02-12,4.727,75.117704,61.676457,102.000200,0
2026-02-13,4.671,68.693188,64.015367,78.048830,0
2026-02-24,4.716,73.185640,67.072125,85.412670,0
2026-02-25,4.750,76.568204,70.237485,89.229644,0
2026-02-26,4.735,75.847057,72.107342,83.326486,0
2026-02-27,4.725,67.982122,70.732269,62.481829,-1
2026-03-02,4.739,66.943036,69.469191,61.890726,0
2026-03-03,4.678,48.463499,62.467294,20.455910,0



Strategy A (K<20 / K>50)
signal
-1     124
 0    1839
 1      10
Name: count, dtype: int64


,Close,K,D,J,signal
Date,,,,,
2026-02-11,4.723,66.572661,54.955833,89.806316,0
2026-02-12,4.727,75.117704,61.676457,102.000200,0
2026-02-13,4.671,68.693188,64.015367,78.048830,0
2026-02-24,4.716,73.185640,67.072125,85.412670,0
2026-02-25,4.750,76.568204,70.237485,89.229644,0
2026-02-26,4.735,75.847057,72.107342,83.326486,0
2026-02-27,4.725,67.982122,70.732269,62.481829,-1
2026-03-02,4.739,66.943036,69.469191,61.890726,0
2026-03-03,4.678,48.463499,62.467294,20.455910,0



Strategy B (K<15 / K>60)
signal
-1      92
 0    1878
 1       3
Name: count, dtype: int64


,Close,K,D,J,signal
Date,,,,,
2026-02-11,4.723,66.572661,54.955833,89.806316,0
2026-02-12,4.727,75.117704,61.676457,102.000200,0
2026-02-13,4.671,68.693188,64.015367,78.048830,0
2026-02-24,4.716,73.185640,67.072125,85.412670,0
2026-02-25,4.750,76.568204,70.237485,89.229644,0
2026-02-26,4.735,75.847057,72.107342,83.326486,0
2026-02-27,4.725,67.982122,70.732269,62.481829,-1
2026-03-02,4.739,66.943036,69.469191,61.890726,0
2026-03-03,4.678,48.463499,62.467294,20.455910,0


## 4) 策略回测模块：`backtest_strategy(fund_code)`
回测设置（对三套KDJ策略一致）：
- 只做多，不加杠杆，不做空
- 按信号次日执行（避免未来函数）
- 仓位状态：空仓才买、满仓才卖
- 佣金统一：`0.00006`，单笔最小 `0.2`

输出对比（共4条曲线/列）：
- Original（K<20 / K>60）
- Strategy A（K<15 / K>70）
- Strategy B（K<30 / K>50）
- Buy & Hold

指标包含：累计收益、最大回撤、胜率、夏普比率

In [38]:
def _max_drawdown(equity: pd.Series) -> float:
    """Compute max drawdown from an equity curve."""
    s = equity.dropna()
    if s.empty:
        return np.nan
    peak = s.cummax()
    dd = s / peak - 1
    return float(dd.min())


def _sharpe_ratio(daily_ret: pd.Series, risk_free_rate: float = 0.02) -> float:
    """Annualized Sharpe ratio using daily returns."""
    r = daily_ret.dropna()
    if len(r) < 2:
        return np.nan
    rf_daily = risk_free_rate / 252
    ex = r - rf_daily
    vol = ex.std(ddof=0)
    if vol == 0 or np.isnan(vol):
        return np.nan
    return float(np.sqrt(252) * ex.mean() / vol)


def _calc_trade_win_rate(trade_action: pd.Series, close: pd.Series) -> float:
    """Win rate based on executed trades (buy=1, sell=-1)."""
    entries = close[trade_action == 1]
    exits = close[trade_action == -1]
    if entries.empty or exits.empty:
        return np.nan

    exit_idx = iter(exits.index)
    current_exit = next(exit_idx, None)
    wins, trades = 0, 0

    for ent_dt, ent_px in entries.items():
        while current_exit is not None and current_exit <= ent_dt:
            current_exit = next(exit_idx, None)
        if current_exit is None:
            break
        ex_px = float(close.loc[current_exit])
        if ent_px > 0:
            trades += 1
            if ex_px / ent_px - 1 > 0:
                wins += 1

    if trades == 0:
        return np.nan
    return float(wins / trades)


def _run_stateful_backtest(df_signal: pd.DataFrame, initial_cash: float = 10000.0, commission_rate: float = 0.00006, min_fee: float = 0.2) -> pd.DataFrame:
    """Run stateful long-only backtest.

    State constraints
    - Buy only when empty position.
    - Sell only when fully invested.
    """
    df = df_signal.dropna().copy()
    close = df['Close'].astype(float)

    # Execute signal on next bar to avoid look-ahead bias.
    exec_signal = df['signal'].shift(1).fillna(0).astype(int)

    cash = float(initial_cash)
    shares = 0.0

    asset_list, fee_list, hold_list, action_list = [], [], [], []

    for px, sig in zip(close.values, exec_signal.values):
        fee = 0.0
        action = 0

        # Empty -> Full on buy signal.
        if sig == 1 and shares == 0:
            notional = cash
            fee = max(notional * commission_rate, min_fee)
            invest = max(cash - fee, 0.0)
            shares = invest / px if px > 0 else 0.0
            cash = cash - invest - fee
            action = 1

        # Full -> Empty on sell signal.
        elif sig == -1 and shares > 0:
            notional = shares * px
            fee = max(notional * commission_rate, min_fee)
            cash = cash + notional - fee
            shares = 0.0
            action = -1

        asset = cash + shares * px
        asset_list.append(asset)
        fee_list.append(fee)
        hold_list.append(1 if shares > 0 else 0)
        action_list.append(action)

    bt = df.copy()
    bt['exec_signal'] = exec_signal
    bt['trade_action'] = action_list
    bt['holding_state'] = hold_list
    bt['fee_paid'] = fee_list
    bt['current_asset'] = asset_list
    bt['equity'] = bt['current_asset'] / initial_cash
    bt['daily_return'] = bt['equity'].pct_change().fillna(0.0)

    bt['bh_equity'] = close / close.iloc[0]
    bt['bh_return'] = bt['bh_equity'].pct_change().fillna(0.0)
    return bt


def _strategy_metrics(bt: pd.DataFrame) -> Dict[str, float]:
    """Calculate key metrics from one strategy backtest dataframe."""
    return {
        'Total Return': float(bt['equity'].iloc[-1] - 1),
        'Max Drawdown': _max_drawdown(bt['equity']),
        'Win Rate': _calc_trade_win_rate(bt['trade_action'], bt['Close']),
        'Sharpe Ratio': _sharpe_ratio(bt['daily_return']),
    }


def backtest_strategy(fund_code: str, start_date: str = '2018-01-01', end_date: Optional[str] = None, initial_cash: float = 10000.0) -> Dict[str, object]:
    """Run multi-strategy KDJ backtests and return comparison outputs.

    Included strategies
    - Strategy A: buy K<15, sell K>70
    - Strategy B: buy K<30, sell K>50
    """
    raw = load_market_data(fund_code, start_date=start_date, end_date=end_date)
    df_kdj = calculate_kdj(raw)
    strategy_signal_map = build_strategy_signals(df_kdj)

    strategy_bt_map: Dict[str, pd.DataFrame] = {}
    metrics_map: Dict[str, Dict[str, float]] = {}

    for name, sig_df in strategy_signal_map.items():
        bt = _run_stateful_backtest(sig_df, initial_cash=initial_cash, commission_rate=0.00006, min_fee=0.2)
        strategy_bt_map[name] = bt
        metrics_map[name] = _strategy_metrics(bt)

    # Buy & Hold baseline (same benchmark for all strategies).
    first_bt = next(iter(strategy_bt_map.values()))
    bh_metrics = {
        'Total Return': float(first_bt['bh_equity'].iloc[-1] - 1),
        'Max Drawdown': _max_drawdown(first_bt['bh_equity']),
        'Win Rate': np.nan,
        'Sharpe Ratio': _sharpe_ratio(first_bt['bh_return']),
    }

    metrics_index = ['Total Return', 'Max Drawdown', 'Win Rate', 'Sharpe Ratio']
    metrics = pd.DataFrame({'Metric': metrics_index})
    for name in strategy_bt_map.keys():
        metrics[name] = [metrics_map[name][m] for m in metrics_index]
    metrics['Buy & Hold'] = [bh_metrics[m] for m in metrics_index]

    return {
        'strategy_bt_map': strategy_bt_map,
        'metrics': metrics,
        'base_data': df_kdj,
    }

In [39]:
result = backtest_strategy(FUND_CODE, START_DATE, END_DATE, initial_cash=10000.0)
strategy_bt_map = result['strategy_bt_map']
metrics = result['metrics']

# 指标对比表（含回撤）
display(metrics)

# 展示两套策略各自回测明细尾部
for name, bt in strategy_bt_map.items():
    print(f'\n{name} - tail')
    display(bt[['Close', 'signal', 'exec_signal', 'trade_action', 'holding_state', 'equity', 'bh_equity']].tail(10))

,Metric,Original (K<20 / K>60),Strategy A (K<20 / K>50),Strategy B (K<15 / K>60),Buy & Hold
0,Total Return,0.149613,0.257798,0.197427,0.092974
1,Max Drawdown,-0.253814,-0.112538,-0.148332,-0.451007
2,Win Rate,0.714286,0.571429,0.666667,NaN
3,Sharpe Ratio,0.020737,0.163080,0.077916,0.054646



Original (K<20 / K>60) - tail


,Close,signal,exec_signal,trade_action,holding_state,equity,bh_equity
Date,,,,,,,
2026-02-13,4.671,0,0,0,0,1.149613,1.093911
2026-02-24,4.716,0,0,0,0,1.149613,1.104450
2026-02-25,4.750,0,0,0,0,1.149613,1.112412
2026-02-26,4.735,0,0,0,0,1.149613,1.108899
2026-02-27,4.725,-1,0,0,0,1.149613,1.106557
2026-03-02,4.739,0,-1,0,0,1.149613,1.109836
2026-03-03,4.678,0,0,0,0,1.149613,1.095550
2026-03-04,4.610,0,0,0,0,1.149613,1.079625
2026-03-05,4.654,0,0,0,0,1.149613,1.089930



Strategy A (K<20 / K>50) - tail


,Close,signal,exec_signal,trade_action,holding_state,equity,bh_equity
Date,,,,,,,
2026-02-13,4.671,0,0,0,0,1.257798,1.093911
2026-02-24,4.716,0,0,0,0,1.257798,1.104450
2026-02-25,4.750,0,0,0,0,1.257798,1.112412
2026-02-26,4.735,0,0,0,0,1.257798,1.108899
2026-02-27,4.725,-1,0,0,0,1.257798,1.106557
2026-03-02,4.739,0,-1,0,0,1.257798,1.109836
2026-03-03,4.678,0,0,0,0,1.257798,1.095550
2026-03-04,4.610,0,0,0,0,1.257798,1.079625
2026-03-05,4.654,0,0,0,0,1.257798,1.089930



Strategy B (K<15 / K>60) - tail


,Close,signal,exec_signal,trade_action,holding_state,equity,bh_equity
Date,,,,,,,
2026-02-13,4.671,0,0,0,0,1.197427,1.093911
2026-02-24,4.716,0,0,0,0,1.197427,1.104450
2026-02-25,4.750,0,0,0,0,1.197427,1.112412
2026-02-26,4.735,0,0,0,0,1.197427,1.108899
2026-02-27,4.725,-1,0,0,0,1.197427,1.106557
2026-03-02,4.739,0,-1,0,0,1.197427,1.109836
2026-03-03,4.678,0,0,0,0,1.197427,1.095550
2026-03-04,4.610,0,0,0,0,1.197427,1.079625
2026-03-05,4.654,0,0,0,0,1.197427,1.089930


## 5) 收益曲线可视化模块
这里对比 4 条收益曲线：
- Original（K<20 / K>60）
- Strategy A（K<20 / K>20）
- Strategy B（K<15 / K>60）
- Buy & Hold

图表支持：拖动、缩放、悬停查看。

In [40]:
def plot_performance(strategy_bt_map: Dict[str, pd.DataFrame], title: str) -> go.Figure:
    """Plot multi-strategy equity curves with Buy & Hold baseline."""
    if not strategy_bt_map:
        raise ValueError('strategy_bt_map is empty')

    fig = go.Figure()

    # Plot each strategy equity curve.
    for name, bt in strategy_bt_map.items():
        if 'equity' not in bt.columns:
            raise ValueError(f'missing equity column in {name}')
        fig.add_trace(go.Scatter(x=bt.index, y=bt['equity'], mode='lines', name=name))

    # Plot one shared Buy & Hold baseline.
    first_bt = next(iter(strategy_bt_map.values()))
    if 'bh_equity' not in first_bt.columns:
        raise ValueError('missing bh_equity column')
    fig.add_trace(go.Scatter(x=first_bt.index, y=first_bt['bh_equity'], mode='lines', name='Buy & Hold', line=dict(dash='dash')))

    fig.update_layout(
        template='plotly_white',
        title=title,
        height=560,
        hovermode='x unified',
        yaxis_title='Equity Curve'
    )
    return fig


plot_performance(strategy_bt_map, title=f'{FUND_CODE} - KDJ Strategy Comparison vs Buy & Hold').show()

## 6) 总结与风险提示
### 总结
1. KDJ适合识别短期动能变化，尤其是金叉/死叉阶段。
2. 它反应快。但也更容易被噪音干扰。
3. 仅用一个指标就能搭建基础策略，但稳定性有限。

### 局限性与失效场景
- 在极端单边趋势中（2020年8月2022年6月），KDJ可能持续钝化，信号变差。
- 震荡噪音较强时，可能频繁交易并增加成本。
- 预警区的参数阈值（20/60）不是万能，需要按品种、市场周期、个人风险偏好调整。

### 重要声明
最后说一句。我自己也在学习阶段。如果你有好的策略。或者想测试某个基金或某个指标。可以在评论区留言。我可以帮大家做回测分析。一起研究。谢谢大家。

技术指标不是预测未来的水晶球，它只是帮助我们更有结构地观察市场行为。
回测表现不代表未来收益，投资有风险，决策需谨慎。
